# 04 - Fakturoid Integration

This notebook tests integration with Fakturoid API for submitting invoices.


In [1]:
from src.fakturoid_client import FakturoidClient
from src.config import config
import json


In [2]:
# Initialize Fakturoid client
fakturoid = FakturoidClient(config)

print("Fakturoid Client initialized")
print(f"Account: {config.fakturoid.account_slug}")
print(f"Base URL: {config.fakturoid.base_url}")


Fakturoid Client initialized
Account: bohemiafalconstudio
Base URL: https://app.fakturoid.cz/api/v3


In [3]:
# Test connection - get account info
try:
    account_info = fakturoid.get_account_info()
    print("✓ Successfully connected to Fakturoid")
    print(f"\nAccount Info:")
    print(json.dumps(account_info, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"✗ Failed to connect to Fakturoid: {e}")


✓ Successfully connected to Fakturoid

Account Info:
{
  "subdomain": "bohemiafalconstudio",
  "plan": "Na maximum",
  "plan_price": 519,
  "plan_paid_users": 0,
  "invoice_email": "zverina.pavel@seznam.cz",
  "phone": "",
  "web": "",
  "name": "Bohemia Falcon Studio, s.r.o.",
  "full_name": null,
  "registration_no": "24167185",
  "vat_no": "CZ24167185",
  "local_vat_no": null,
  "vat_mode": "vat_payer",
  "vat_price_mode": "without_vat",
  "street": "Plynární 1032/29",
  "city": "Praha 7",
  "zip": "17000",
  "country": "CZ",
  "currency": "CZK",
  "unit_name": "",
  "vat_rate": 21,
  "displayed_note": "Společnost je zapsána v obchodním rejstříku vedeném Městským soudem v Praze oddíl C, vložka 184888.",
  "invoice_note": null,
  "due": 14,
  "invoice_language": "cz",
  "invoice_payment_method": null,
  "invoice_proforma": false,
  "invoice_hide_bank_account_for_payments": null,
  "fixed_exchange_rate": false,
  "invoice_selfbilling": false,
  "default_estimate_type": null,
  "send_o

In [4]:
# List existing subjects (suppliers)
try:
    subjects = fakturoid.list_subjects()
    print(f"Found {len(subjects)} subjects in Fakturoid:")
    for subject in subjects[:5]:  # Show first 5
        print(f"  - {subject.get('name')} (ID: {subject.get('id')})")
    if len(subjects) > 5:
        print(f"  ... and {len(subjects) - 5} more")
except Exception as e:
    print(f"✗ Failed to list subjects: {e}")


Found 40 subjects in Fakturoid:
  - 123RF Limited (ID: 14468485)
  - 1. e-shop s.r.o. (ID: 14468465)
  - 1. holešovická restaurační s.r.o. (ID: 14468464)
  - 3D Magic LLC (ID: 14468551)
  - AB papír s.r.o. (ID: 14458962)
  ... and 35 more


In [5]:
# Test invoice submission using submit_expense method
from src.ai_extractor import InvoiceData

# Create sample invoice data
sample_invoice = InvoiceData(
    invoice_number="TEST-001",
    issue_date="2024-10-08",
    supplier_name="Test Supplier s.r.o.",
    total_amount=1210.0,  # Including 21% VAT
    due_date="2024-10-22",
    supplier_address="Testovací 123, Praha",
    supplier_ico="12345678",
    supplier_dic="CZ12345678",
    currency="CZK",
    variable_symbol="001",
    notes="Test invoice for API integration"
)

print("Sample invoice data:")
print(json.dumps(sample_invoice.model_dump(), indent=2, ensure_ascii=False))

# Submit to Fakturoid (automatically creates supplier if needed)
print("\n" + "="*60)
print("SUBMITTING TO FAKTUROID...")
print("="*60)

try:
    result = fakturoid.submit_expense(sample_invoice, auto_create_subject=True)
    print(f"\n✓ Expense created successfully!")
    print(f"Expense ID: {result.get('id')}")
    print(f"Number: {result.get('number')}")
    print(f"Supplier: {result.get('supplier_name')}")
    print(f"Total: {result.get('total')} {result.get('currency')}")
    print(f"\nView in Fakturoid: {result.get('html_url')}")
except Exception as e:
    print(f"\n✗ Failed to submit: {e}")
    import traceback
    traceback.print_exc()


Sample invoice data:
{
  "invoice_number": "TEST-001",
  "issue_date": "2024-10-08",
  "supplier_name": "Test Supplier s.r.o.",
  "total_amount": 1210.0,
  "due_date": "2024-10-22",
  "variable_symbol": "001",
  "supplier_address": "Testovací 123, Praha",
  "supplier_ico": "12345678",
  "supplier_dic": "CZ12345678",
  "currency": "CZK",
  "tax_amount": null,
  "line_items": null,
  "notes": "Test invoice for API integration",
  "confidence": null,
  "source_file": null
}

SUBMITTING TO FAKTUROID...

✓ Expense created successfully!
Expense ID: 3467496
Number: FP20240187
Supplier: Test Supplier s.r.o.
Total: 1210.0 CZK

View in Fakturoid: https://app.fakturoid.cz/bohemiafalconstudio/expenses/3467496


In [ ]:
# List recent expense invoices
try:
    invoices = fakturoid.list_expense_invoices(limit=10)
    print(f"Recent expense invoices ({len(invoices)}):")
    for inv in invoices:
        print(f"  - {inv.get('number')} | {inv.get('supplier_name')} | {inv.get('total')} {inv.get('currency')}")
except Exception as e:
    print(f"✗ Failed to list invoices: {e}")


In [ ]:
# Test with REAL invoice from PDF
from src.ai_extractor import AIExtractor
from src.document_processor import DocumentProcessor

print("Testing with real invoice from PDF...")
print("="*60)

# Get first invoice file
doc_processor = DocumentProcessor(config.directories.invoices)
files = doc_processor.list_invoice_files()

if files:
    test_file = files[0]
    print(f"\n📄 Processing: {test_file.name}")
    
    # Extract data using AI
    ai_extractor = AIExtractor(config)
    invoice_data_dict = ai_extractor.extract_invoice_data(test_file)
    
    # Convert to InvoiceData model
    invoice_data = InvoiceData(**invoice_data_dict)
    
    print(f"\n✓ Extracted data:")
    print(f"  Supplier: {invoice_data.supplier_name}")
    print(f"  Invoice #: {invoice_data.invoice_number}")
    print(f"  Date: {invoice_data.issue_date}")
    print(f"  Amount: {invoice_data.total_amount} {invoice_data.currency}")
    
    # Ask before submitting
    print(f"\n{'='*60}")
    print("Ready to submit to Fakturoid")
    print(f"{'='*60}")
    
    # UNCOMMENT to actually submit:
    # try:
    #     result = fakturoid.submit_expense(invoice_data, auto_create_subject=True)
    #     print(f"\n✓ Expense created successfully!")
    #     print(f"Expense ID: {result.get('id')}")
    #     print(f"Number: {result.get('number')}")
    #     print(f"Supplier: {result.get('supplier_name')}")
    #     print(f"Total: {result.get('total')} {result.get('currency')}")
    #     print(f"\nView in Fakturoid: {result.get('html_url')}")
    # except Exception as e:
    #     print(f"\n✗ Failed to submit: {e}")
    #     import traceback
    #     traceback.print_exc()
else:
    print("No invoice files found in data/invoices/")
